# DPO — Paper-to-Code Mock (Colab)

**Paper:** Direct Preference Optimization: Your Language Model is Secretly a Reward Model (Rafailov et al., 2023) — https://arxiv.org/abs/2305.18290

Medium-hard mock (~40 min). Fill in the `dpo_loss` stub, run the preference-alignment toy task, then the sanity checks. The big idea: RLHF **without a reward model and without an RL loop** — just one supervised loss on preferred-vs-dispreferred pairs. Reference solution in the last cell.

In [ ]:
import math
import torch
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the DPO loss (YOUR TASK)

For each (chosen `y_w`, rejected `y_l`) pair, given per-sequence log-probs under the policy and the FROZEN reference:

L = -log sigma( beta * [ (logp_pol_chosen - logp_ref_chosen) - (logp_pol_rejected - logp_ref_rejected) ] )

No reward model, no RL. Use `F.logsigmoid` for stability.

In [ ]:
def dpo_loss(logp_pol_chosen, logp_pol_rejected,
             logp_ref_chosen, logp_ref_rejected, beta=0.1):
    # TODO: pol_logratio = logp_pol_chosen - logp_pol_rejected
    # TODO: ref_logratio = logp_ref_chosen - logp_ref_rejected
    # TODO: logits = beta * (pol_logratio - ref_logratio)
    # TODO: return -F.logsigmoid(logits).mean()
    raise NotImplementedError("fill me in")


def implicit_reward(logp_pol, logp_ref, beta=0.1):
    """r(y) = beta * log(pi_theta / pi_ref). Used only to READ OFF the learned reward."""
    return beta * (logp_pol - logp_ref)

## 2. Demonstrate the benefit (preference-alignment toy task)

A tiny discrete policy: a categorical over 6 "responses", per prompt, with learnable logits. The reference is a frozen copy of the initial (uniform) policy. We feed it preference pairs from a known ranking and train with `dpo_loss` only. After training, the **implicit reward should order the responses like the preferences** — no reward model trained, no RL loop.

In [ ]:
class ToyPolicy(torch.nn.Module):
    """pi(y|x): a categorical over n_responses, per prompt. Logits ARE the policy."""
    def __init__(self, n_prompts, n_responses):
        super().__init__()
        self.logits = torch.nn.Parameter(torch.zeros(n_prompts, n_responses))  # uniform

    def logprobs(self, prompt_idx):
        return F.log_softmax(self.logits[prompt_idx], dim=-1)


def run_toy_task(seed=0, beta=0.1, steps=400):
    torch.manual_seed(seed)
    n_prompts, n_responses = 2, 6
    policy = ToyPolicy(n_prompts, n_responses)

    # Frozen reference = a copy of the INITIAL policy.
    ref = ToyPolicy(n_prompts, n_responses)
    ref.load_state_dict(policy.state_dict())
    for p in ref.parameters():
        p.requires_grad_(False)

    # Known rankings (best -> worst); build all chosen>rejected pairs from them.
    rankings = {0: [0, 1, 2, 3, 4, 5], 1: [5, 4, 3, 2, 1, 0]}
    pairs = [(pr, o[a], o[b]) for pr, o in rankings.items()
             for a in range(len(o)) for b in range(a + 1, len(o))]
    prompt_t = torch.tensor([p for p, _, _ in pairs])
    chosen_t = torch.tensor([c for _, c, _ in pairs])
    reject_t = torch.tensor([r for _, _, r in pairs])

    before = policy.logprobs(torch.arange(n_prompts)).exp().detach()

    opt = torch.optim.Adam(policy.parameters(), lr=0.05)
    for _ in range(steps):
        lp_pol = policy.logprobs(prompt_t)
        with torch.no_grad():
            lp_ref = ref.logprobs(prompt_t)
        loss = dpo_loss(lp_pol.gather(1, chosen_t[:, None]).squeeze(1),
                        lp_pol.gather(1, reject_t[:, None]).squeeze(1),
                        lp_ref.gather(1, chosen_t[:, None]).squeeze(1),
                        lp_ref.gather(1, reject_t[:, None]).squeeze(1), beta=beta)
        opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        after = policy.logprobs(torch.arange(n_prompts)).exp()
        rewards = implicit_reward(policy.logprobs(torch.arange(n_prompts)),
                                  ref.logprobs(torch.arange(n_prompts)), beta=beta)
    return dict(before=before, after=after, rewards=rewards,
                rankings=rankings, ref=ref, policy=policy)


out = run_toy_task(seed=0)
for pr, order in out["rankings"].items():
    ranked = sorted(range(6), key=lambda y: out["rewards"][pr, y].item(), reverse=True)
    print(f"prompt {pr}: top response prob {out['before'][pr, order[0]]:.3f} -> "
          f"{out['after'][pr, order[0]]:.3f}   reward ranking {ranked}  "
          f"(target {order}, match={ranked == order})")

## 3. Sanity checks

In [ ]:
# 1) at init (pi == pi_ref) the loss is -log sigma(0) = log 2
z = torch.zeros(8)
assert abs(dpo_loss(z, z, z, z, beta=0.1).item() - math.log(2)) < 1e-6

# 2) reference policy is frozen (no grad, never moved)
ref = out["ref"]
assert all(not p.requires_grad for p in ref.parameters())
assert torch.equal(ref.logits, torch.zeros_like(ref.logits))

# 3) gradient flows to the policy
pol = ToyPolicy(2, 6)
with torch.no_grad(): pol.logits[0, 0] += 0.3
lp, lpr = pol.logprobs(torch.tensor([0])), torch.zeros(1, 6).log_softmax(-1)
dpo_loss(lp[:, 0], lp[:, 1], lpr[:, 0], lpr[:, 1], beta=0.1).backward()
assert pol.logits.grad is not None and pol.logits.grad.abs().sum() > 0

# 4) bigger margin lowers loss; swapping chosen/rejected raises it
def L(pc, pr): return dpo_loss(torch.tensor([pc]), torch.tensor([pr]),
                               torch.tensor([0.0]), torch.tensor([0.0]), beta=1.0).item()
assert L(3.0, -3.0) < L(0.0, 0.0) < L(-3.0, 3.0) and L(3.0, -3.0) < 0.01

# 5) after training, implicit-reward ranking matches the preferences
for pr, order in out["rankings"].items():
    ranked = sorted(range(6), key=lambda y: out["rewards"][pr, y].item(), reverse=True)
    assert ranked == order
    assert out["after"][pr, order[0]] > out["before"][pr, order[0]]
# reproducible with a fixed seed
assert torch.allclose(out["after"], run_toy_task(seed=0)["after"], atol=1e-6)

# 6) implicit reward scales linearly with beta
lp_, lr_ = torch.tensor([math.log(0.5)]), torch.tensor([math.log(0.25)])
assert torch.allclose(implicit_reward(lp_, lr_, beta=0.2),
                      2 * implicit_reward(lp_, lr_, beta=0.1), atol=1e-6)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def dpo_loss_solution(logp_pol_chosen, logp_pol_rejected,
                      logp_ref_chosen, logp_ref_rejected, beta=0.1):
    pol_logratio = logp_pol_chosen - logp_pol_rejected
    ref_logratio = logp_ref_chosen - logp_ref_rejected
    logits = beta * (pol_logratio - ref_logratio)
    return -F.logsigmoid(logits).mean()